# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

In [ ]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

In [ ]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [ ]:
print(embeddings.shape, embeddings.dtype)

In [ ]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [ ]:
!pip install faiss-cpu

In [ ]:
# código base para FAISS
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

D, I = index.search(query_vec, k=10)

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?

> Se usó cosine porque la colección se creó con distance cosine y esta métrica compara la dirección de los vectores sin depender de la magnitud, lo que funciona mejor para búsqueda semántica en textos generados por modelos como E5.

- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?

> En Qdrant fue más fácil porque la metadata se guarda como payload junto al vector y se puede filtrar de forma nativa en la misma consulta, mientras que en FAISS normalmente se necesita un mapeo externo de IDs.

- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?

> El tiempo crece a medida que k es mayor porque deben recuperarse y ordenarse más vectores, aunque con un dataset de 80k el aumento es pequeño para top 10 o 20.

In [ ]:
# Paso 0: Instalación

!pip install qdrant-client

In [ ]:
# Paso 1: Conectar con Qdrant

# Importar librerías
from qdrant_client import QdrantClient
from qdrant_client.http import models

client = QdrantClient(":memory:")

print("Qdrant corriendo en modo local (in-memory).")


In [ ]:
# Paso 2: Crear colección

from qdrant_client.http import models

# Dimensión de los embeddings (D)
dim = embeddings.shape[1]

# Crear colección si no existe
if not client.collection_exists("wiki_chunks"):
    client.create_collection(
        collection_name="wiki_chunks",
        vectors_config=models.VectorParams(
            size=dim,
            distance=models.Distance.COSINE   # usamos cosine porque normalizamos los embeddings
        )
    )

print("Colección 'wiki_chunks' creada o ya existente.")


In [ ]:
# Paso 3: Insertar embeddings + metadata

from qdrant_client.http import models

def upload_in_batches(client, embeddings, df, batch_size=500):
    for start in range(0, len(embeddings), batch_size):
        end = start + batch_size
        batch_points = []
        for i, (vec, row) in enumerate(zip(embeddings[start:end], df.iloc[start:end].itertuples()), start=start):
            batch_points.append(models.PointStruct(
                id=i,
                vector=vec.tolist(),
                payload={
                    "doc_id": row.doc_id,
                    "chunk_id": row.chunk_id,
                    "text": row.text
                }
            ))
        client.upsert(
            collection_name="wiki_chunks",
            points=batch_points
        )
        print(f"Inserted batch {start}–{end}")

upload_in_batches(client, embeddings, chunks_df, batch_size=500)

In [ ]:
# Paso 4: Función de búsqueda

from qdrant_client.http import models

def qdrant_search(query_embedding, k=5):
    # 1. Cambio: El parámetro se llama 'query', no 'query_vector'
    results = client.query_points(
        collection_name="wiki_chunks",
        query=query_embedding[0].tolist(),  # <--- Cambiado aquí
        query_filter=None,
        limit=k,
        with_payload=True,
        with_vectors=False
    )

    output = []
    # 2. Cambio: query_points devuelve un objeto QueryResponse,
    # hay que iterar sobre su atributo '.points'
    for r in results.points:  # <--- Cambiado aquí
        output.append({
            "id": r.id,
            "score": r.score,
            "text": r.payload.get("text"),
            "metadata": {
                "doc_id": r.payload.get("doc_id"),
                "chunk_id": r.payload.get("chunk_id")
            }
        })
    return output


In [ ]:
# Ejemplo de consulta

res = qdrant_search(query_vec, k=5)
for r in res:
    print(r["score"], r["text"][:120], r["metadata"])



0.870348534612154 Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro {'doc_id': 1391, 'chunk_id': 0}
0.861800414009939 Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter {'doc_id': 1, 'chunk_id': 0}
0.8401016188686827 ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries {'doc_id': 1391, 'chunk_id': 1}
0.8391333416603377 ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply  {'doc_id': 5067, 'chunk_id': 1}
0.8385888618440084 is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa {'doc_id': 9888, 'chunk_id': 2}


## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?

> Se modificó nlist en el índice IVF y nprobe en la búsqueda; nlist define los clústeres y nprobe cuántos revisar, controlando el equilibrio entre rapidez y exactitud.

- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?

> Al comparar índice flat exacto con IVF aproximado aparecen diferencias en posiciones bajas del top debido a que nprobe puede dejar fuera un clúster con vectores relevantes.

In [ ]:
# Paso 1: Instalar Milvus y el cliente en Colab

!pip install "pymilvus[milvus_lite]"

In [ ]:
# Paso 2: Conectar a Milvus

from pymilvus import MilvusClient

# Esto crea un archivo local 'milvus_wikipedia.db' en tu Colab
client_milvus = MilvusClient("milvus_wikipedia.db")
COLLECTION_NAME = "wiki_chunks"

print("Milvus Lite conectado localmente.")

In [ ]:
# Si existe, la borramos para empezar de cero
if client_milvus.has_collection(COLLECTION_NAME):
    client_milvus.drop_collection(COLLECTION_NAME)

# Creamos la colección definiendo solo la dimensión y métrica.
# MilvusClient habilita "Dynamic Schema" automáticamente para guardar texto/metadata.
client_milvus.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=embeddings.shape[1],  # Dimensión de tus embeddings (768 para E5)
    metric_type="COSINE"            # Métrica de similitud
)

print(f"Colección '{COLLECTION_NAME}' creada localmente.")



In [ ]:
# 3. Preparar datos e Insertar por lotes (FIX PARA EVITAR RESOURCE_EXHAUSTED)
data_to_insert = []
print("Preparando datos...")

# Recorremos tus embeddings y dataframe para armar los diccionarios
for i in range(len(embeddings)):
    data_to_insert.append({
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": chunks_df.iloc[i]["text"],
        "doc_id": int(chunks_df.iloc[i]["doc_id"]),
        "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
    })

BATCH_SIZE = 2000  # Insertar de 2000 en 2000 para no saturar la memoria/red
total_records = len(data_to_insert)
print(f"Iniciando inserción de {total_records} vectores en lotes de {BATCH_SIZE}...")

from tqdm.auto import tqdm

# Bucle para insertar por partes
for i in tqdm(range(0, total_records, BATCH_SIZE)):
    batch = data_to_insert[i : i + BATCH_SIZE]
    res = client_milvus.insert(
        collection_name=COLLECTION_NAME,
        data=batch
    )

print("Inserción completada exitosamente.")

In [ ]:
# 4. Función de búsqueda ajustada a MilvusClient
def milvus_search(query_embedding, k=5):
    # Asegurar que el vector sea una lista de floats
    if hasattr(query_embedding, "flatten"):
        query_vector = query_embedding.flatten().tolist()
    else:
        query_vector = query_embedding

    search_res = client_milvus.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        limit=k,
        # 'output_fields' le dice qué metadatos devolver
        output_fields=["text", "doc_id", "chunk_id"],
        search_params={"metric_type": "COSINE", "params": {}}
    )

    # Formatear salida limpia
    output = []
    for hit in search_res[0]:
        output.append({
            "id": hit["id"],
            "score": hit["distance"],
            "text": hit["entity"].get("text"),
            "metadata": {
                "doc_id": hit["entity"].get("doc_id"),
                "chunk_id": hit["entity"].get("chunk_id")
            }
        })
    return output

# Prueba
resultados = milvus_search(query_vec, k=5)
for r in resultados:
    print(f"Score: {r['score']:.4f} | Text: {r['text'][:100]}...")

Score: 0.8703 | Text: Battery tester A battery tester is an electronic device intended for testing the state of an electri...
Score: 0.8618 | Text: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
Score: 0.8401 | Text: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
Score: 0.8391 | Text: ils. One was connected via a series resistor to the battery supply. The second was connected to the ...
Score: 0.8386 | Text: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...


## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?

> Weaviate maneja clases y objetos que encapsulan datos y vectores de manera flexible, distinto al modelo relacional rígido de tabla con filas.

- ¿Cómo describirías el trade-off de complejidad vs expresividad?

> Es más complejo al inicio por definir esquema, pero permite filtros tipados y búsquedas híbridas mucho más expresivas.



In [ ]:
# 1. Instalación
!pip install -U weaviate-client

import weaviate
from weaviate.classes.config import Property, DataType

# 2. Conexión a Weaviate Embedded (Local en Colab)
# Esto descarga y ejecuta el binario de Weaviate en segundo plano
client_weaviate = weaviate.connect_to_embedded()

print("Weaviate Embedded está corriendo.")

In [ ]:
# 2: Definir Esquema

# Nombre de la colección
COLLECTION_NAME = "WikiChunk"

# Si ya existe, la borramos para empezar limpio
if client_weaviate.collections.exists(COLLECTION_NAME):
    client_weaviate.collections.delete(COLLECTION_NAME)

# 3. Crear colección con esquema
# Definimos las propiedades que queremos guardar como metadata/texto
client_weaviate.collections.create(
    name=COLLECTION_NAME,
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="doc_id", data_type=DataType.INT),
        Property(name="chunk_id", data_type=DataType.INT),
    ],
    # IMPORTANTE: No configuramos un vectorizador automático porque
    # nosotros traeremos nuestros propios embeddings de E5.
    vectorizer_config=None
)

print(f"Colección '{COLLECTION_NAME}' creada.")

In [ ]:
# 3: Insertar Objetos (Batch)

import weaviate.classes as wvc
from tqdm.auto import tqdm

# Referencia a la colección
collection = client_weaviate.collections.get(COLLECTION_NAME)

print("Iniciando inserción masiva en Weaviate...")

# Usamos el context manager para batching automático
with collection.batch.dynamic() as batch:
    for i in tqdm(range(len(embeddings))):
        # Preparamos las propiedades (metadata + texto)
        properties = {
            "text": chunks_df.iloc[i]["text"],
            "doc_id": int(chunks_df.iloc[i]["doc_id"]),
            "chunk_id": int(chunks_df.iloc[i]["chunk_id"])
        }

        # Añadimos el objeto pasando explícitamente el vector
        batch.add_object(
            properties=properties,
            vector=embeddings[i].tolist()  # Weaviate espera listas, no numpy arrays
        )

# Verificación rápida
count = collection.aggregate.over_all(total_count=True).total_count
print(f"Inserción finalizada. Total de objetos en Weaviate: {count}")

In [ ]:
# 4: Función de Búsqueda y Prueba

from weaviate.classes.query import MetadataQuery

def weaviate_search(query_embedding, k=5):
    # Asegurar que el vector de consulta sea una lista simple
    if hasattr(query_embedding, "flatten"):
        vec = query_embedding.flatten().tolist()
    else:
        vec = query_embedding

    collection = client_weaviate.collections.get(COLLECTION_NAME)

    # 4. Consultar por similitud
    response = collection.query.near_vector(
        near_vector=vec,
        limit=k,
        return_metadata=MetadataQuery(distance=True) # Pedimos la distancia
    )

    # Formatear la salida según lo solicitado
    output = []
    for obj in response.objects:
        output.append({
            "id": obj.uuid,  # Weaviate genera UUIDs automáticamente
            "score": obj.metadata.distance, # En Weaviate devuelve distancia (menor es mejor)
            "text": obj.properties["text"],
            "metadata": {
                "doc_id": obj.properties["doc_id"],
                "chunk_id": obj.properties["chunk_id"]
            }
        })
    return output

# --- Prueba ---
print(f"Buscando: '{query_text}'")
# query_vec viene de las partes anteriores del notebook
res_weaviate = weaviate_search(query_vec, k=5)

for r in res_weaviate:
    print(f"Distancia: {r['score']:.4f} | Texto: {r['text'][:100]}...")

Buscando: 'Battery measuring'
Distancia: 0.1297 | Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electri...
Distancia: 0.1382 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
Distancia: 0.1599 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
Distancia: 0.1609 | Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the ...
Distancia: 0.1614 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?

> Chroma fue más simple porque no exigió esquemas ni servicios pesados y su uso en Python es directo para prototipos.

- ¿Qué limitaciones ves para un sistema en producción?

> Puede presentar problemas de escalabilidad y concurrencia al usar SQLite local o memoria, además de menos monitoreo que bases dedicadas.



In [ ]:
# 1. Instalación
!pip install chromadb

import chromadb
# Importamos la excepción específica para manejar el caso de "no encontrado"
from chromadb.errors import NotFoundError

# 2. Inicializar Cliente Efímero (In-Memory)
chroma_client = chromadb.Client()

# Crear la colección
# Intentamos borrarla primero por si ya existe de una ejecución anterior
try:
    chroma_client.delete_collection(name="wiki_collection")
except (ValueError, NotFoundError):
    # Si falla porque no existe (NotFoundError) o por valor (ValueError), no pasa nada
    pass
except Exception as e:
    # Cualquier otro error lo imprimimos por si acaso
    print(f"Nota: No se pudo borrar la colección previa: {e}")

# Creamos la colección nueva
collection_chroma = chroma_client.create_collection(name="wiki_collection")

print("Cliente Chroma listo y colección 'wiki_collection' creada.")

In [ ]:
# 3. Preparar listas de datos
# Chroma espera listas de Python, no arrays de Numpy ni DataFrames
print("Preparando listas de datos...")
all_embeddings = embeddings.tolist()
all_documents = chunks_df['text'].tolist()
all_ids = [str(i) for i in range(len(chunks_df))] # Los IDs deben ser strings

# Preparamos la metadata como lista de diccionarios
all_metadatas = [
    {"doc_id": int(row.doc_id), "chunk_id": int(row.chunk_id)}
    for row in chunks_df.itertuples()
]

# 4. Inserción por lotes
batch_size = 5000
total_docs = len(all_ids)

print(f"Insertando {total_docs} documentos en Chroma...")

for i in tqdm(range(0, total_docs, batch_size)):
    end_idx = min(i + batch_size, total_docs)

    collection_chroma.add(
        embeddings=all_embeddings[i:end_idx],
        documents=all_documents[i:end_idx],
        metadatas=all_metadatas[i:end_idx],
        ids=all_ids[i:end_idx]
    )

print("Inserción en Chroma completada.")

In [ ]:
def chroma_search(query_embedding, k=5):
    # Convertir el embedding de numpy a lista plana
    if hasattr(query_embedding, "flatten"):
        query_list = query_embedding.flatten().tolist()
    else:
        query_list = query_embedding

    # Ejecutar consulta
    results = collection_chroma.query(
        query_embeddings=[query_list],
        n_results=k,
        # Especificamos qué datos queremos de vuelta
        include=["documents", "metadatas", "distances"]
    )

    # Formatear la salida (Chroma devuelve listas de listas)
    formatted_results = []

    # Iteramos sobre el primer (y único) vector de consulta
    for i in range(len(results['ids'][0])):
        formatted_results.append({
            "id": results['ids'][0][i],
            "score": results['distances'][0][i], # Distancia L2 por defecto
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })

    return formatted_results

# --- Prueba ---
print(f"Buscando en Chroma: '{query_text}'")
res_chroma = chroma_search(query_vec, k=5)

for r in res_chroma:
    # Nota: Chroma usa distancia Euclídea (L2) por defecto si no se normaliza,
    # pero como usamos E5 normalizado, el ranking es equivalente a Coseno.
    print(f"Distancia: {r['score']:.4f} | Texto: {r['text'][:100]}...")

Buscando en Chroma: 'Battery measuring'
Distancia: 0.2593 | Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electri...
Distancia: 0.2764 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
Distancia: 0.3198 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
Distancia: 0.3217 | Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the ...
Distancia: 0.3228 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?

> Es la más transparente porque la búsqueda se hace con SQL visible y depurable como cualquier consulta tradicional.

- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?

> Permite joins, filtros y agregaciones combinando similitud vectorial con reglas de negocio sin separar datos.

- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?

> Postgres no es nativo para vectores y a gran escala suele rendir peor y consumir más RAM que Milvus o Qdrant con sharding y HNSW optimizados.



In [ ]:
# 1. Instalar dependencias del sistema y PostgreSQL
!apt-get update
!apt-get -y install postgresql-14 postgresql-server-dev-14 build-essential

# 2. Compilar e instalar la extensión pgvector
!git clone https://github.com/pgvector/pgvector.git
%cd pgvector
!make
!make install
%cd ..

# 3. Instalar librerías de Python
!pip install pgvector psycopg2-binary

In [ ]:
# 4. Iniciar el servicio de PostgreSQL
!service postgresql start

# 5. Configurar usuario y base de datos
# Creamos un usuario 'colab' con contraseña 'colab'
!sudo -u postgres psql -c "CREATE USER colab WITH PASSWORD 'colab';"
!sudo -u postgres psql -c "CREATE DATABASE vector_db OWNER colab;"
!sudo -u postgres psql -c "ALTER USER colab WITH SUPERUSER;"

# Habilitamos la extensión vector en la base de datos
!sudo -u postgres psql -d vector_db -c "CREATE EXTENSION IF NOT EXISTS vector;"

print("PostgreSQL corriendo y extensión vector activada.")

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector
import numpy as np

# Conexión a la DB local
conn = psycopg2.connect(
    dbname="vector_db",
    user="colab",
    password="colab",
    host="localhost"
)

# Registrar adaptador de pgvector para manejar arrays numpy
register_vector(conn)
cur = conn.cursor()

# 1. Crear Tabla
# Asumimos dimensión 768 (E5-base), ajusta si usas otro modelo
DIMENSION = embeddings.shape[1]

cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute(f"""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        text TEXT,
        doc_id INTEGER,
        chunk_id INTEGER,
        embedding vector({DIMENSION})
    );
""")
conn.commit()
print("Tabla 'documents' creada.")

# 2. Insertar Datos (Batch)
print("Preparando datos para inserción...")

# Preparamos una lista de tuplas
data_rows = [
    (
        chunks_df.iloc[i]["text"],
        int(chunks_df.iloc[i]["doc_id"]),
        int(chunks_df.iloc[i]["chunk_id"]),
        embeddings[i] # pgvector maneja el numpy array gracias a register_vector
    )
    for i in range(len(embeddings))
]

insert_query = """
    INSERT INTO documents (text, doc_id, chunk_id, embedding)
    VALUES %s
"""

print(f"Insertando {len(data_rows)} filas en PostgreSQL...")
execute_values(cur, insert_query, data_rows, page_size=1000)
conn.commit()

print("Inserción completada.")

In [ ]:
def pgvector_search(query_embedding, k=5):
    # 1. Convertir a Numpy si es lista
    if not isinstance(query_embedding, np.ndarray):
        query_vec = np.array(query_embedding)
    else:
        query_vec = query_embedding

    # 2. FIX: Aplanar el vector para que sea 1D (768,) en lugar de 2D (1, 768)
    # pgvector lanzará "expected ndim to be 1" si no hacemos esto.
    if query_vec.ndim > 1:
        query_vec = query_vec.flatten()

    # Consulta SQL (operador <=> es distancia coseno)
    search_sql = """
    SELECT id, (embedding <=> %s) as distance, text, doc_id, chunk_id
    FROM documents
    ORDER BY distance ASC
    LIMIT %s;
    """

    try:
        # Ahora query_vec tiene la forma correcta
        cur.execute(search_sql, (query_vec, k))
        rows = cur.fetchall()

        results = []
        for r in rows:
            results.append({
                "id": r[0],
                "score": r[1],
                "text": r[2],
                "metadata": {"doc_id": r[3], "chunk_id": r[4]}
            })
        return results
    except Exception as e:
        conn.rollback()
        print(f"Error en consulta: {e}")
        return []

# --- Prueba ---
print(f"Buscando en PostgreSQL: '{query_text}'")
# Generamos el embedding de nuevo por si acaso
query_vec = embed_query(query_text)

res_pg = pgvector_search(query_vec, k=5)

for r in res_pg:
    print(f"Distancia: {r['score']:.4f} | Texto: {r['text'][:100]}...")

Buscando en PostgreSQL: 'Battery measuring'
Distancia: 0.1297 | Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electri...
Distancia: 0.1382 | Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
Distancia: 0.1599 | Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
Distancia: 0.1609 | Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the ...
Distancia: 0.1614 | Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...
